# CS2Q2

## Data Preparation and Cleaning

We first get the data. Note that this part is the same as in CS2Q1.

In [4]:
import pandas as pd

data_path = "../data/wildfire/merged_with_wsei.csv"
df_raw = pd.read_csv(data_path)

df = df_raw.drop(columns=["Date"]) # drop the duplicate date column

# rename columns
df = df.rename(columns={
    "Station ID": "station_id",
    "Date": "date",
    "Latitude": "lat",
    "Longitude": "lon",
    "AQI": "aqi"
}).copy()

# parse date and create calendar features
df["date"] = pd.to_datetime(df["date"])
df["dow"] = df["date"].dt.day_name().str[:3]   # similar to wday(label = TRUE)
df["month"] = df["date"].dt.month
df["doy"] = df["date"].dt.dayofyear
df["year"] = df["date"].dt.year

# sort and create next-day AQI target
df = df.sort_values(["station_id", "date"]).copy()
df["aqi_next"] = df.groupby("station_id")["aqi"].shift(-1)

# drop rows without target
df = df[df["aqi_next"].notna()].copy()

# optional: reset index
df = df.reset_index(drop=True)

Then as in Q1, we 

In [6]:
import pandas as pd
import numpy as np

# wildfire variables
wsei_vars = ["wsei_hfi_k0", "wsei_hfi_k1", "wsei_hfi_k2", "wsei_hfi_k3"]

# basic covariates
# covars = ["MeanTemp", "TotalPrecip", "MaxTemp", "MinTemp", "TempRange", "TotalTraffic", "wind_dir_deg"]
# Note that we drop the wind_dir_deg as too many missing values
covars = ["MeanTemp", "TotalPrecip", "MaxTemp", "MinTemp", "TempRange", "TotalTraffic"]

# columns to keep
keep_cols = ["station_id", "date", "aqi_next", "dow", "doy", "month", "year"] + wsei_vars + covars

# keep only needed columns
df_sub = df[keep_cols].copy()

# sample size before dropping NA
n_before = len(df_sub)

# missing count by column
missing_counts = df_sub.isna().sum().sort_values(ascending=False)

# drop rows with any missing values in selected columns
df_m = df_sub.dropna().copy()

# sample size after dropping NA
n_after = len(df_m)
n_dropped = n_before - n_after
drop_ratio = n_dropped / n_before

print(f"Rows before dropna: {n_before:,}")
print(f"Rows after dropna:  {n_after:,}")
print(f"Rows dropped:       {n_dropped:,}")
print(f"Drop ratio:         {drop_ratio:.2%}")

print("\nMissing counts by selected column:")
print(missing_counts)

# standardize continuous predictors
scale_cols = wsei_vars + covars

for col in scale_cols:
    mean = df_m[col].mean()
    std = df_m[col].std()
    df_m[col + "_z"] = (df_m[col] - mean) / std

Rows before dropna: 33,496
Rows after dropna:  18,057
Rows dropped:       15,439
Drop ratio:         46.09%

Missing counts by selected column:
TotalPrecip     14569
TempRange       13371
MeanTemp        13371
MinTemp         13370
MaxTemp         13369
TotalTraffic     3506
wsei_hfi_k2         0
wsei_hfi_k3         0
station_id          0
date                0
wsei_hfi_k0         0
year                0
month               0
doy                 0
dow                 0
aqi_next            0
wsei_hfi_k1         0
dtype: int64


In this step we need to deal with missing values. We notice that some of the stations has missing values for all the weather / traffic covariates. We need to figure out how many stations has weather / traffic data.

In [12]:
# wildfire variables
wsei_vars = ["wsei_hfi_k0", "wsei_hfi_k1", "wsei_hfi_k2", "wsei_hfi_k3"]

# basic covariates
# covars = ["MeanTemp", "TotalPrecip", "MaxTemp", "MinTemp", "TempRange", "TotalTraffic", "wind_dir_deg"]
# Note that we drop the wind_dir_deg as too many missing values
covars = ["MeanTemp", "TotalPrecip", "TempRange", "TotalTraffic"]

# columns to keep
keep_cols = ["station_id", "date", "aqi_next", "dow", "doy", "month", "year"] + wsei_vars + covars

# keep only needed columns and drop rows with missing values in the main covariates
df_no_wind = df[keep_cols].copy()
df_no_wind_drop = df_no_wind.dropna().copy()

print("Rows after dropna (no wind):", len(df_no_wind_drop))
print("Stations after dropna (no wind):", df_no_wind_drop["station_id"].nunique())

# Count the percentage of rows with missing values in each kept stations
station_total = df_no_wind.groupby("station_id").size().rename("n_total")
station_kept_no_wind = df_no_wind_drop.groupby("station_id").size().rename("n_kept_no_wind")


station_summary = pd.concat(
    [station_total, station_kept_no_wind],
    axis=1
).fillna(0)

station_summary["n_kept_no_wind"] = station_summary["n_kept_no_wind"].astype(int)
station_summary["keep_ratio_no_wind"] = station_summary["n_kept_no_wind"] / station_summary["n_total"]

kept_stations_no_wind = station_summary[station_summary["n_kept_no_wind"] > 0].copy()
print(
    kept_stations_no_wind[
        ["n_total", "n_kept_no_wind", "keep_ratio_no_wind"]
    ].sort_values("keep_ratio_no_wind")
)


Rows after dropna (no wind): 18057
Stations after dropna (no wind): 22
            n_total  n_kept_no_wind  keep_ratio_no_wind
station_id                                             
15026           881             799            0.906924
45027           879             798            0.907850
12008           883             813            0.920725
12016           883             814            0.921857
51001           883             857            0.970555
63203           880             855            0.971591
29118           880             861            0.978409
29000           882             863            0.978458
44008           884             865            0.978507
28028           881             867            0.984109
49005           882             869            0.985261
56051           884             873            0.987557
35125           882             876            0.993197
21005           880             875            0.994318
52023           883             8

Note that only 22 stations are kept after dropna, and station 63200 has only two rows after dropping, so we will keep only 21 stations. We then fill up the missing values by IterativeImputer, which is a more sophisticated imputation method that models each feature with missing values as a function of other features.
Also, we will split training and testing data by 80-20 split. And to avoid data leakage, we will fit the imputer only on the training data and then transform both training and testing data.

In [13]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
import pandas as pd

# Keep only stations with a sufficient number of observations after dropping rows with missing values in the main covariates
station_counts = df_no_wind_drop.groupby("station_id").size()
valid_stations = station_counts[station_counts >= 100].index

# Get the data back from the original dataframe, but only for the valid stations
df_imp = df[df["station_id"].isin(valid_stations)].copy()

# Keep only rows with a valid next-day AQI target
df_imp = df_imp[df_imp["aqi_next"].notna()].copy()

# Sort by station and date before splitting
df_imp = df_imp.sort_values(["station_id", "date"]).reset_index(drop=True)

# Select variables to impute
# Exclude wind_dir_deg from the main specification due to severe missingness
impute_cols = ["MeanTemp", "TotalPrecip", "TempRange", "TotalTraffic"]

# Create a global time-based 80/20 split
dates = pd.to_datetime(df_imp["date"])
cutoff = dates.quantile(0.8)

train_idx = df_imp["date"] <= cutoff
test_idx = df_imp["date"] > cutoff

df_train = df_imp.loc[train_idx].copy()
df_test = df_imp.loc[test_idx].copy()

print("Train shape before imputation:", df_train.shape)
print("Test shape before imputation:", df_test.shape)
print("Cutoff date:", cutoff)

# Fit the imputer on each station's training subset,
# then apply the same fitted imputer to both train and test subsets
train_parts = []
test_parts = []

for station_id in sorted(df_imp["station_id"].unique()):
    train_station = df_train[df_train["station_id"] == station_id].sort_values("date").copy()
    test_station = df_test[df_test["station_id"] == station_id].sort_values("date").copy()

    # Skip stations that do not appear in the training set
    if train_station.empty:
        continue

    imputer = IterativeImputer(random_state=0) # Here random_state is set for reproducibility (similar to set.seed in R)
    imputer.fit(train_station[impute_cols])

    train_station.loc[:, impute_cols] = imputer.transform(train_station[impute_cols])

    if not test_station.empty:
        test_station.loc[:, impute_cols] = imputer.transform(test_station[impute_cols])

    train_parts.append(train_station)
    test_parts.append(test_station)

df_train_imp = pd.concat(train_parts, axis=0).sort_values(["station_id", "date"]).reset_index(drop=True)
df_test_imp = pd.concat(test_parts, axis=0).sort_values(["station_id", "date"]).reset_index(drop=True)

# Combine the imputed train and test sets if a full imputed dataset is needed
df_imp = pd.concat([df_train_imp, df_test_imp], axis=0).sort_values(["station_id", "date"]).reset_index(drop=True)

print("Train shape after imputation:", df_train_imp.shape)
print("Test shape after imputation:", df_test_imp.shape)
print("Full imputed dataset shape:", df_imp.shape)

print("\nRemaining missing values in train:")
print(df_train_imp[impute_cols].isna().sum())

print("\nRemaining missing values in test:")
print(df_test_imp[impute_cols].isna().sum())

# Make sure that the stations in training and test sets are the same as in the original split
train_stations = set(df_train_imp["station_id"].unique())
test_stations = set(df_test_imp["station_id"].unique())

print("\nNumber of train stations:", len(train_stations))
print("Number of test stations:", len(test_stations))
print("Stations only in train:", sorted(train_stations - test_stations))
print("Stations only in test:", sorted(test_stations - train_stations))

Train shape before imputation: (14827, 41)
Test shape before imputation: (3696, 41)
Cutoff date: 2024-04-01 00:00:00
Train shape after imputation: (14827, 41)
Test shape after imputation: (3696, 41)
Full imputed dataset shape: (18523, 41)

Remaining missing values in train:
MeanTemp        0
TotalPrecip     0
TempRange       0
TotalTraffic    0
dtype: int64

Remaining missing values in test:
MeanTemp        0
TotalPrecip     0
TempRange       0
TotalTraffic    0
dtype: int64

Number of train stations: 21
Number of test stations: 21
Stations only in train: []
Stations only in test: []


After the data cleaning, then we build the model. 

## Model Building and Evaluation
### OLS 
Note that from Q1, we get that OLS is then best model for one-day-ahead AQI prediction. We include it as a baseline model here. We will use the same covariates as in Q1, which are the current-day AQI and the weather / traffic covariates.

Also, we will get two versions of OLS, one with only weather & traffic covariates, and one add additional WSEI covariates.

Also, in the first version of OLS, we use the original data, without log transformatiom.

In [23]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Define feature sets
baseline_cont_vars = ["MeanTemp", "TotalPrecip", "TempRange", "TotalTraffic", "aqi"]
wsei_vars = ["wsei_hfi_k0", "wsei_hfi_k1", "wsei_hfi_k2", "wsei_hfi_k3"]
scale_cols = baseline_cont_vars + wsei_vars

# Create working copies
train_ols = df_train_imp.copy()
test_ols = df_test_imp.copy()

# Standardize continuous variables using training-set statistics only
train_means = train_ols[scale_cols].mean()
train_stds = train_ols[scale_cols].std().replace(0, 1)

for col in scale_cols:
    train_ols[f"{col}_z"] = (train_ols[col] - train_means[col]) / train_stds[col]
    test_ols[f"{col}_z"] = (test_ols[col] - train_means[col]) / train_stds[col]

# Ensure categorical variables have consistent types
train_ols["station_id"] = train_ols["station_id"].astype(str)
test_ols["station_id"] = test_ols["station_id"].astype(str)
train_ols["dow"] = train_ols["dow"].astype(str)
test_ols["dow"] = test_ols["dow"].astype(str)

# Define helper functions
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def evaluate_regression(y_true, y_pred, label):
    return pd.Series({
        "model": label,
        "RMSE": rmse(y_true, y_pred),
        "MAE": mean_absolute_error(y_true, y_pred)
    })

def evaluate_top_decile(test_df, pred_col, target_col="aqi_next", label="Model"):
    cutoff = test_df[target_col].quantile(0.9)
    df_tail = test_df[test_df[target_col] >= cutoff].copy()
    return pd.Series({
        "model": label,
        "n_top_decile": len(df_tail),
        "top_decile_RMSE": rmse(df_tail[target_col], df_tail[pred_col]),
        "top_decile_MAE": mean_absolute_error(df_tail[target_col], df_tail[pred_col])
    })

# Define OLS formulas
baseline_formula = """
aqi_next ~
aqi_z + MeanTemp_z + TotalPrecip_z + TempRange_z + TotalTraffic_z + 
doy + C(dow) + C(station_id)
"""

augmented_formula = """
aqi_next ~
aqi_z + MeanTemp_z + TotalPrecip_z + TempRange_z + TotalTraffic_z + 
wsei_hfi_k0_z + wsei_hfi_k1_z + wsei_hfi_k2_z + wsei_hfi_k3_z +
doy + C(dow) + C(station_id)
"""

# Fit baseline OLS
ols_baseline = smf.ols(formula=baseline_formula, data=train_ols).fit()
train_ols["pred_baseline"] = ols_baseline.predict(train_ols)
test_ols["pred_baseline"] = ols_baseline.predict(test_ols)

# Fit augmented OLS
ols_augmented = smf.ols(formula=augmented_formula, data=train_ols).fit()
train_ols["pred_augmented"] = ols_augmented.predict(train_ols)
test_ols["pred_augmented"] = ols_augmented.predict(test_ols)

# Evaluate overall performance
baseline_train_metrics = evaluate_regression(
    train_ols["aqi_next"], train_ols["pred_baseline"], "OLS Baseline - Train"
)
baseline_test_metrics = evaluate_regression(
    test_ols["aqi_next"], test_ols["pred_baseline"], "OLS Baseline - Test"
)
aug_train_metrics = evaluate_regression(
    train_ols["aqi_next"], train_ols["pred_augmented"], "OLS + Wildfire - Train"
)
aug_test_metrics = evaluate_regression(
    test_ols["aqi_next"], test_ols["pred_augmented"], "OLS + Wildfire - Test"
)

# Evaluate top-decile performance on the test set
baseline_tail_metrics = evaluate_top_decile(
    test_ols, pred_col="pred_baseline", target_col="aqi_next", label="OLS Baseline - Test Top Decile"
)
aug_tail_metrics = evaluate_top_decile(
    test_ols, pred_col="pred_augmented", target_col="aqi_next", label="OLS + Wildfire - Test Top Decile"
)

# Build comparison tables
overall_comparison = pd.DataFrame([
    baseline_train_metrics,
    baseline_test_metrics,
    aug_train_metrics,
    aug_test_metrics
])

tail_comparison = pd.DataFrame([
    baseline_tail_metrics,
    aug_tail_metrics
])

# Extract wildfire coefficient table
coef_table = pd.DataFrame({
    "coef": ols_augmented.params,
    "p_value": ols_augmented.pvalues
})

wildfire_coef_table = coef_table.loc[
    ["wsei_hfi_k0_z", "wsei_hfi_k1_z", "wsei_hfi_k2_z", "wsei_hfi_k3_z"]
].copy()

# Print key outputs
print("Overall performance comparison:")
display(overall_comparison)

print("Top-decile performance comparison:")
display(tail_comparison)

print("Wildfire coefficient table:")
display(wildfire_coef_table)

print("Baseline OLS summary:")
print(ols_baseline.summary())

print("Augmented OLS summary:")
print(ols_augmented.summary())

Overall performance comparison:


,model,RMSE,MAE
0,OLS Baseline - Train,15.093071,9.696389
1,OLS Baseline - Test,11.726270,8.682664
2,OLS + Wildfire - Train,15.028890,9.683900
3,OLS + Wildfire - Test,11.704901,8.697979


Top-decile performance comparison:


,model,n_top_decile,top_decile_RMSE,top_decile_MAE
0,OLS Baseline - Test Top Decile,392,22.470814,19.058420
1,OLS + Wildfire - Test Top Decile,392,22.638192,19.316394


Wildfire coefficient table:


,coef,p_value
wsei_hfi_k0_z,0.685502,0.000473
wsei_hfi_k1_z,-1.224405,0.000023
wsei_hfi_k2_z,1.344146,0.000045
wsei_hfi_k3_z,0.640891,0.010907


Baseline OLS summary:
                            OLS Regression Results                            
Dep. Variable:               aqi_next   R-squared:                       0.365
Model:                            OLS   Adj. R-squared:                  0.364
Method:                 Least Squares   F-statistic:                     266.1
Date:                Sat, 07 Mar 2026   Prob (F-statistic):               0.00
Time:                        20:34:50   Log-Likelihood:                -61283.
No. Observations:               14827   AIC:                         1.226e+05
Df Residuals:                   14794   BIC:                         1.229e+05
Df Model:                          32                                         
Covariance Type:            nonrobust                                         
                             coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------
Interc

We can see that adding the wildfire features show almost no improvement to the overall accuracy. However, the model with wildfire features shows worse performance in predicting extreme AQI days. 

Note that we include the wildfire features mainly for the extreme days, but what we get now does not match our expectation. We now change the wildfire features to a log-transformed version, and check the results.

In [24]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Define feature sets
baseline_cont_vars = ["MeanTemp", "TotalPrecip", "TempRange", "TotalTraffic", "aqi"]
wsei_vars_raw = ["wsei_hfi_k0", "wsei_hfi_k1", "wsei_hfi_k2", "wsei_hfi_k3"]
wsei_vars_log = [f"{col}_log1p" for col in wsei_vars_raw]

# Create working copies
train_ols = df_train_imp.copy()
test_ols = df_test_imp.copy()

# Create log-transformed wildfire features
for raw_col, log_col in zip(wsei_vars_raw, wsei_vars_log):
    train_ols[log_col] = np.log1p(train_ols[raw_col])
    test_ols[log_col] = np.log1p(test_ols[raw_col])

# Standardize continuous variables using training-set statistics only
scale_cols = baseline_cont_vars + wsei_vars_log
train_means = train_ols[scale_cols].mean()
train_stds = train_ols[scale_cols].std().replace(0, 1)

for col in scale_cols:
    train_ols[f"{col}_z"] = (train_ols[col] - train_means[col]) / train_stds[col]
    test_ols[f"{col}_z"] = (test_ols[col] - train_means[col]) / train_stds[col]

# Ensure categorical variables have consistent types
train_ols["station_id"] = train_ols["station_id"].astype(str)
test_ols["station_id"] = test_ols["station_id"].astype(str)
train_ols["dow"] = train_ols["dow"].astype(str)
test_ols["dow"] = test_ols["dow"].astype(str)

# Define helper functions
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def evaluate_regression(y_true, y_pred, label):
    return pd.Series({
        "model": label,
        "RMSE": rmse(y_true, y_pred),
        "MAE": mean_absolute_error(y_true, y_pred)
    })

def evaluate_top_decile(test_df, pred_col, target_col="aqi_next", label="Model"):
    cutoff = test_df[target_col].quantile(0.9)
    df_tail = test_df[test_df[target_col] >= cutoff].copy()
    return pd.Series({
        "model": label,
        "n_top_decile": len(df_tail),
        "top_decile_RMSE": rmse(df_tail[target_col], df_tail[pred_col]),
        "top_decile_MAE": mean_absolute_error(df_tail[target_col], df_tail[pred_col])
    })

# Define OLS formulas
baseline_formula = """
aqi_next ~
aqi_z + MeanTemp_z + TotalPrecip_z + TempRange_z + TotalTraffic_z +
doy + C(dow) + C(station_id)
"""

log_wildfire_formula = """
aqi_next ~
aqi_z + MeanTemp_z + TotalPrecip_z + TempRange_z + TotalTraffic_z +
wsei_hfi_k0_log1p_z + wsei_hfi_k1_log1p_z + wsei_hfi_k2_log1p_z + wsei_hfi_k3_log1p_z +
doy + C(dow) + C(station_id)
"""

# Fit baseline OLS
ols_baseline = smf.ols(formula=baseline_formula, data=train_ols).fit()
train_ols["pred_baseline"] = ols_baseline.predict(train_ols)
test_ols["pred_baseline"] = ols_baseline.predict(test_ols)

# Fit log-wildfire OLS
ols_log_wildfire = smf.ols(formula=log_wildfire_formula, data=train_ols).fit()
train_ols["pred_log_wildfire"] = ols_log_wildfire.predict(train_ols)
test_ols["pred_log_wildfire"] = ols_log_wildfire.predict(test_ols)

# Evaluate overall performance
baseline_train_metrics = evaluate_regression(
    train_ols["aqi_next"], train_ols["pred_baseline"], "OLS Baseline - Train"
)
baseline_test_metrics = evaluate_regression(
    test_ols["aqi_next"], test_ols["pred_baseline"], "OLS Baseline - Test"
)
log_train_metrics = evaluate_regression(
    train_ols["aqi_next"], train_ols["pred_log_wildfire"], "OLS + log(Wildfire) - Train"
)
log_test_metrics = evaluate_regression(
    test_ols["aqi_next"], test_ols["pred_log_wildfire"], "OLS + log(Wildfire) - Test"
)

# Evaluate top-decile performance on the test set
baseline_tail_metrics = evaluate_top_decile(
    test_ols, pred_col="pred_baseline", target_col="aqi_next", label="OLS Baseline - Test Top Decile"
)
log_tail_metrics = evaluate_top_decile(
    test_ols, pred_col="pred_log_wildfire", target_col="aqi_next", label="OLS + log(Wildfire) - Test Top Decile"
)

# Build comparison tables
overall_comparison = pd.DataFrame([
    baseline_train_metrics,
    baseline_test_metrics,
    log_train_metrics,
    log_test_metrics
])

tail_comparison = pd.DataFrame([
    baseline_tail_metrics,
    log_tail_metrics
])

# Extract wildfire coefficient table
coef_table = pd.DataFrame({
    "coef": ols_log_wildfire.params,
    "p_value": ols_log_wildfire.pvalues
})

wildfire_coef_table = coef_table.loc[
    ["wsei_hfi_k0_log1p_z", "wsei_hfi_k1_log1p_z", "wsei_hfi_k2_log1p_z", "wsei_hfi_k3_log1p_z"]
].copy()

# Print key outputs
print("Overall performance comparison:")
display(overall_comparison)

print("Top-decile performance comparison:")
display(tail_comparison)

print("Log-wildfire coefficient table:")
display(wildfire_coef_table)

print("Baseline OLS summary:")
print(ols_baseline.summary())

print("OLS + log(Wildfire) summary:")
print(ols_log_wildfire.summary())

Overall performance comparison:


,model,RMSE,MAE
0,OLS Baseline - Train,15.093071,9.696389
1,OLS Baseline - Test,11.726270,8.682664
2,OLS + log(Wildfire) - Train,14.820623,9.639451
3,OLS + log(Wildfire) - Test,12.061681,9.075287


Top-decile performance comparison:


,model,n_top_decile,top_decile_RMSE,top_decile_MAE
0,OLS Baseline - Test Top Decile,392,22.470814,19.058420
1,OLS + log(Wildfire) - Test Top Decile,392,22.417705,18.822187


Log-wildfire coefficient table:


,coef,p_value
wsei_hfi_k0_log1p_z,2.385915,2.833137e-26
wsei_hfi_k1_log1p_z,1.310325,1.148605e-07
wsei_hfi_k2_log1p_z,0.136721,5.832726e-01
wsei_hfi_k3_log1p_z,-0.314310,1.548412e-01


Baseline OLS summary:
                            OLS Regression Results                            
Dep. Variable:               aqi_next   R-squared:                       0.365
Model:                            OLS   Adj. R-squared:                  0.364
Method:                 Least Squares   F-statistic:                     266.1
Date:                Sat, 07 Mar 2026   Prob (F-statistic):               0.00
Time:                        20:35:33   Log-Likelihood:                -61283.
No. Observations:               14827   AIC:                         1.226e+05
Df Residuals:                   14794   BIC:                         1.229e+05
Df Model:                          32                                         
Covariance Type:            nonrobust                                         
                             coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------
Interc

We can see that applying log transform has cause a significant decrease to the overall performance of the model, but we it do improve the performance in predicting extreme AQI days.

This suggests that the predictive value of wildfire exposure in this dataset is carried partly by its high-intensity spikes, which are overly compressed by the log transform.

So for OLS, we conclude that the wildfire features do show signal in predicting AQI, with raw data consider to be more useful than the log transformed data in overall performance, and log data more useful in predicting extreme AQI days. However, the problem comes from Case 1, that OLS does not perform well in predicting extreme AQI days, still exists, even with the wildfire features.

## Neural Network
The OLS results confirm that wildfire features contain some predictive signal, but they also show that a linear model cannot effectively improve extreme-day forecasting. We therefore turn to a more flexible neural network model to examine whether nonlinear relationships between wildfire exposure and AQI can be better captured, especially for extreme AQI days.

In [36]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Create working copies
train_nn = df_train_imp.copy()
test_nn = df_test_imp.copy()

# Ensure correct data types
train_nn["station_id"] = train_nn["station_id"].astype(str)
test_nn["station_id"] = test_nn["station_id"].astype(str)
train_nn["dow"] = train_nn["dow"].astype(str)
test_nn["dow"] = test_nn["dow"].astype(str)

# Define feature sets
baseline_num_vars = ["aqi", "MeanTemp", "TotalPrecip", "TempRange", "TotalTraffic", "doy"]
wsei_vars = ["wsei_hfi_k0", "wsei_hfi_k1", "wsei_hfi_k2", "wsei_hfi_k3"]
cat_vars = ["station_id", "dow"]

baseline_features = baseline_num_vars + cat_vars
augmented_features = baseline_num_vars + wsei_vars + cat_vars

target_col = "aqi_next"

# Define helper functions
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def evaluate_regression(y_true, y_pred, label):
    return pd.Series({
        "model": label,
        "RMSE": rmse(y_true, y_pred),
        "MAE": mean_absolute_error(y_true, y_pred)
    })

def evaluate_top_decile(test_df, pred_col, target_col="aqi_next", label="Model"):
    cutoff = test_df[target_col].quantile(0.9)
    df_tail = test_df[test_df[target_col] >= cutoff].copy()
    return pd.Series({
        "model": label,
        "n_top_decile": len(df_tail),
        "top_decile_RMSE": rmse(df_tail[target_col], df_tail[pred_col]),
        "top_decile_MAE": mean_absolute_error(df_tail[target_col], df_tail[pred_col])
    })

# Define a function to build the preprocessing + MLP pipeline
def build_mlp_pipeline(num_features, cat_features):
    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, num_features),
            ("cat", categorical_transformer, cat_features)
        ]
    )

    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", MLPRegressor(
            random_state=0,
            max_iter=500,
            early_stopping=True,
            validation_fraction=0.1,
            n_iter_no_change=20
        ))
    ])

    return pipeline

# Define a function to tune and evaluate a model
def tune_and_evaluate_mlp(
    train_df,
    test_df,
    feature_cols,
    target_col,
    model_label,
    n_splits=5
):
    X_train = train_df[feature_cols].copy()
    y_train = train_df[target_col].copy()

    X_test = test_df[feature_cols].copy()
    y_test = test_df[target_col].copy()

    num_features = [col for col in feature_cols if col not in cat_vars]
    cat_features = [col for col in feature_cols if col in cat_vars]

    pipeline = build_mlp_pipeline(num_features, cat_features)

    param_grid = {
        "model__hidden_layer_sizes": [(32,), (64,), (64, 32)],
        "model__alpha": [1e-4, 1e-3, 1e-2],
        "model__learning_rate_init": [1e-3, 5e-4]
    }

    tscv = TimeSeriesSplit(n_splits=n_splits)

    grid = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        scoring="neg_root_mean_squared_error",
        cv=tscv,
        n_jobs=-1,
        refit=True,
        verbose=1
    )

    grid.fit(X_train, y_train)

    best_model = grid.best_estimator_

    train_pred = best_model.predict(X_train)
    test_pred = best_model.predict(X_test)

    train_eval = evaluate_regression(y_train, train_pred, f"{model_label} - Train")
    test_eval = evaluate_regression(y_test, test_pred, f"{model_label} - Test")

    test_eval_df = test_df.copy()
    test_eval_df["prediction"] = test_pred
    tail_eval = evaluate_top_decile(
        test_eval_df,
        pred_col="prediction",
        target_col=target_col,
        label=f"{model_label} - Test Top Decile"
    )

    return {
        "grid": grid,
        "best_model": best_model,
        "best_params": grid.best_params_,
        "best_cv_rmse": -grid.best_score_,
        "train_eval": train_eval,
        "test_eval": test_eval,
        "tail_eval": tail_eval,
        "train_pred": train_pred,
        "test_pred": test_pred
    }

# Tune and evaluate the baseline NN
nn_baseline_results = tune_and_evaluate_mlp(
    train_df=train_nn,
    test_df=test_nn,
    feature_cols=baseline_features,
    target_col=target_col,
    model_label="NN Baseline",
    n_splits=5
)

# Tune and evaluate the wildfire-augmented NN
nn_augmented_results = tune_and_evaluate_mlp(
    train_df=train_nn,
    test_df=test_nn,
    feature_cols=augmented_features,
    target_col=target_col,
    model_label="NN + Wildfire",
    n_splits=5
)

# Build comparison tables
nn_overall_comparison = pd.DataFrame([
    nn_baseline_results["train_eval"],
    nn_baseline_results["test_eval"],
    nn_augmented_results["train_eval"],
    nn_augmented_results["test_eval"]
])

nn_tail_comparison = pd.DataFrame([
    nn_baseline_results["tail_eval"],
    nn_augmented_results["tail_eval"]
])

# Print key outputs
print("Baseline best parameters:")
print(nn_baseline_results["best_params"])
print("Baseline best CV RMSE:")
print(nn_baseline_results["best_cv_rmse"])

print("\nAugmented best parameters:")
print(nn_augmented_results["best_params"])
print("Augmented best CV RMSE:")
print(nn_augmented_results["best_cv_rmse"])

print("\nOverall performance comparison:")
display(nn_overall_comparison)

print("Top-decile performance comparison:")
display(nn_tail_comparison)

Fitting 5 folds for each of 18 candidates, totalling 90 fits


c:\Users\bchen\anaconda3\envs\mlfin\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


Fitting 5 folds for each of 18 candidates, totalling 90 fits
Baseline best parameters:
{'model__alpha': 0.001, 'model__hidden_layer_sizes': (32,), 'model__learning_rate_init': 0.0005}
Baseline best CV RMSE:
18.511119231911735

Augmented best parameters:
{'model__alpha': 0.0001, 'model__hidden_layer_sizes': (64, 32), 'model__learning_rate_init': 0.0005}
Augmented best CV RMSE:
17.46293179876211

Overall performance comparison:


,model,RMSE,MAE
0,NN Baseline - Train,13.853603,9.122867
1,NN Baseline - Test,12.300621,9.295328
2,NN + Wildfire - Train,12.271920,8.313734
3,NN + Wildfire - Test,12.782537,9.542219


Top-decile performance comparison:


,model,n_top_decile,top_decile_RMSE,top_decile_MAE
0,NN Baseline - Test Top Decile,392,21.523922,18.380774
1,NN + Wildfire - Test Top Decile,392,22.558778,19.144850


We can see that the NN model shows to have worse performance than OLS in terms of overall RMSE and MAE, this match our result from Case 1, that OLS is the best model for one-day-ahead AQI prediction. 
However, the NN model does show better performance in predicting extreme AQI days, which suggests that the NN model can better predict the extreme AQI days. The NN baseline model shows to have better performance than all the OLS models.

And for adding the wildfire features, we get both overall performance and extreme-day performance to be worse than the NN baseline model.

### LSTM

In [29]:
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Set random seeds for reproducibility
np.random.seed(0)
torch.manual_seed(0)

# Define the computation device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

# Create working copies
train_lstm = df_train_imp.copy()
test_lstm = df_test_imp.copy()

# Ensure correct data types
train_lstm["date"] = pd.to_datetime(train_lstm["date"])
test_lstm["date"] = pd.to_datetime(test_lstm["date"])
train_lstm["station_id"] = train_lstm["station_id"].astype(str)
test_lstm["station_id"] = test_lstm["station_id"].astype(str)
train_lstm["dow"] = train_lstm["dow"].astype(str)
test_lstm["dow"] = test_lstm["dow"].astype(str)

# Sort by station and date
train_lstm = train_lstm.sort_values(["station_id", "date"]).reset_index(drop=True)
test_lstm = test_lstm.sort_values(["station_id", "date"]).reset_index(drop=True)

# Define feature groups
baseline_num_vars = ["aqi", "MeanTemp", "TotalPrecip", "TempRange", "TotalTraffic", "doy"]
wsei_vars = ["wsei_hfi_k0", "wsei_hfi_k1", "wsei_hfi_k2", "wsei_hfi_k3"]
target_col = "aqi_next"

# Create one-hot columns using training-set categories only
station_categories = sorted(train_lstm["station_id"].unique())
dow_categories = sorted(train_lstm["dow"].unique())

train_lstm["station_id"] = pd.Categorical(train_lstm["station_id"], categories=station_categories)
test_lstm["station_id"] = pd.Categorical(test_lstm["station_id"], categories=station_categories)

train_lstm["dow"] = pd.Categorical(train_lstm["dow"], categories=dow_categories)
test_lstm["dow"] = pd.Categorical(test_lstm["dow"], categories=dow_categories)

train_station_dummies = pd.get_dummies(train_lstm["station_id"], prefix="station")
test_station_dummies = pd.get_dummies(test_lstm["station_id"], prefix="station")
train_dow_dummies = pd.get_dummies(train_lstm["dow"], prefix="dow")
test_dow_dummies = pd.get_dummies(test_lstm["dow"], prefix="dow")

test_station_dummies = test_station_dummies.reindex(columns=train_station_dummies.columns, fill_value=0)
test_dow_dummies = test_dow_dummies.reindex(columns=train_dow_dummies.columns, fill_value=0)

station_dummy_cols = train_station_dummies.columns.tolist()
dow_dummy_cols = train_dow_dummies.columns.tolist()

train_lstm = pd.concat([train_lstm, train_station_dummies, train_dow_dummies], axis=1)
test_lstm = pd.concat([test_lstm, test_station_dummies, test_dow_dummies], axis=1)

# Create an inner validation split from the training set using time ordering
val_cutoff = train_lstm["date"].quantile(0.8)
train_inner = train_lstm[train_lstm["date"] <= val_cutoff].copy()
val_inner = train_lstm[train_lstm["date"] > val_cutoff].copy()

print("Train inner shape:", train_inner.shape)
print("Validation shape:", val_inner.shape)
print("Test shape:", test_lstm.shape)
print("Validation cutoff date:", val_cutoff)

# Define feature lists
baseline_feature_cols = baseline_num_vars + station_dummy_cols + dow_dummy_cols
augmented_feature_cols = baseline_num_vars + wsei_vars + station_dummy_cols + dow_dummy_cols

# Standardize continuous variables using train-inner statistics only
def apply_standardization(train_df, val_df, test_df, cols_to_scale):
    train_df = train_df.copy()
    val_df = val_df.copy()
    test_df = test_df.copy()

    means = train_df[cols_to_scale].mean()
    stds = train_df[cols_to_scale].std().replace(0, 1)

    for col in cols_to_scale:
        train_df[col] = (train_df[col] - means[col]) / stds[col]
        val_df[col] = (val_df[col] - means[col]) / stds[col]
        test_df[col] = (test_df[col] - means[col]) / stds[col]

    return train_df, val_df, test_df

# Build fixed-length sequences within each station
def create_sequences(df, feature_cols, target_col, seq_len):
    X_list = []
    y_list = []
    meta_list = []

    for station_id, g in df.groupby("station_id"):
        g = g.sort_values("date").reset_index(drop=True)

        if len(g) < seq_len:
            continue

        X_values = g[feature_cols].to_numpy(dtype=np.float32)
        y_values = g[target_col].to_numpy(dtype=np.float32)
        date_values = g["date"].to_numpy()

        for i in range(seq_len - 1, len(g)):
            X_list.append(X_values[i - seq_len + 1:i + 1])
            y_list.append(y_values[i])
            meta_list.append({
                "station_id": station_id,
                "date": date_values[i],
                target_col: y_values[i]
            })

    X = np.array(X_list, dtype=np.float32)
    y = np.array(y_list, dtype=np.float32)
    meta = pd.DataFrame(meta_list)

    return X, y, meta

# Define a PyTorch dataset
class SequenceDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).view(-1, 1)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Define the LSTM model
class LSTMRegressor(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=1, dropout=0.0):
        super().__init__()

        effective_dropout = dropout if num_layers > 1 else 0.0

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=effective_dropout
        )
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        output, (hidden, cell) = self.lstm(x)
        last_hidden = hidden[-1]
        prediction = self.fc(last_hidden)
        return prediction

# Define evaluation helpers
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def evaluate_regression(y_true, y_pred, label):
    return pd.Series({
        "model": label,
        "RMSE": rmse(y_true, y_pred),
        "MAE": mean_absolute_error(y_true, y_pred)
    })

def evaluate_top_decile(eval_df, pred_col, target_col="aqi_next", label="Model"):
    cutoff = eval_df[target_col].quantile(0.9)
    df_tail = eval_df[eval_df[target_col] >= cutoff].copy()

    return pd.Series({
        "model": label,
        "n_top_decile": len(df_tail),
        "top_decile_RMSE": rmse(df_tail[target_col], df_tail[pred_col]),
        "top_decile_MAE": mean_absolute_error(df_tail[target_col], df_tail[pred_col])
    })

# Define training and prediction functions
def train_lstm_model(
    X_train,
    y_train,
    X_val,
    y_val,
    hidden_size=64,
    num_layers=1,
    dropout=0.0,
    lr=1e-3,
    batch_size=256,
    max_epochs=100,
    patience=10
):
    train_dataset = SequenceDataset(X_train, y_train)
    val_dataset = SequenceDataset(X_val, y_val)

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=False,
        pin_memory=torch.cuda.is_available()
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        pin_memory=torch.cuda.is_available()
    )

    model = LSTMRegressor(
        input_size=X_train.shape[2],
        hidden_size=hidden_size,
        num_layers=num_layers,
        dropout=dropout
    ).to(device)

    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    best_model_state = None
    best_val_loss = np.inf
    patience_counter = 0

    for epoch in range(max_epochs):
        model.train()
        train_losses = []

        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(device, non_blocking=True)
            y_batch = y_batch.to(device, non_blocking=True)

            optimizer.zero_grad()
            pred = model(X_batch)
            loss = criterion(pred, y_batch)

            if torch.isnan(loss):
                print(f"NaN train loss encountered at epoch {epoch + 1}.")
                break

            loss.backward()

            # Apply gradient clipping to stabilize LSTM training
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimizer.step()
            train_losses.append(loss.item())

        if len(train_losses) == 0:
            print("Training stopped because no valid train loss was recorded.")
            break

        model.eval()
        val_losses = []

        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch = X_batch.to(device, non_blocking=True)
                y_batch = y_batch.to(device, non_blocking=True)

                pred = model(X_batch)
                loss = criterion(pred, y_batch)

                if torch.isnan(loss):
                    print(f"NaN validation loss encountered at epoch {epoch + 1}.")
                    break

                val_losses.append(loss.item())

        if len(val_losses) == 0:
            print("Validation stopped because no valid validation loss was recorded.")
            break

        avg_train_loss = np.mean(train_losses)
        avg_val_loss = np.mean(val_losses)

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_model_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1

        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(
                f"Epoch {epoch + 1:03d} | "
                f"Train Loss: {avg_train_loss:.4f} | "
                f"Val Loss: {avg_val_loss:.4f}"
            )

        if patience_counter >= patience:
            print(f"Early stopping triggered at epoch {epoch + 1}.")
            break

    if best_model_state is not None:
        model.load_state_dict(best_model_state)

    return model, best_val_loss

def predict_lstm(model, X):
    model.eval()

    X_tensor = torch.tensor(X, dtype=torch.float32).to(device, non_blocking=True)

    with torch.no_grad():
        preds = model(X_tensor).cpu().numpy().reshape(-1)

    return preds

# Run one LSTM experiment
def run_lstm_experiment(
    train_inner_df,
    val_df,
    test_df,
    feature_cols,
    target_col,
    label,
    seq_len=7,
    hidden_size=64,
    num_layers=1,
    dropout=0.0,
    lr=1e-3,
    batch_size=256,
    max_epochs=100,
    patience=10
):
    cols_to_scale = [col for col in feature_cols if col in baseline_num_vars + wsei_vars]
    train_scaled, val_scaled, test_scaled = apply_standardization(
        train_inner_df,
        val_df,
        test_df,
        cols_to_scale
    )

    X_train, y_train, meta_train = create_sequences(train_scaled, feature_cols, target_col, seq_len)
    X_val, y_val, meta_val = create_sequences(val_scaled, feature_cols, target_col, seq_len)
    X_test, y_test, meta_test = create_sequences(test_scaled, feature_cols, target_col, seq_len)

    print(f"\n{label} sequence shapes:")
    print("X_train:", X_train.shape, "X_val:", X_val.shape, "X_test:", X_test.shape)

    model, best_val_loss = train_lstm_model(
        X_train=X_train,
        y_train=y_train,
        X_val=X_val,
        y_val=y_val,
        hidden_size=hidden_size,
        num_layers=num_layers,
        dropout=dropout,
        lr=lr,
        batch_size=batch_size,
        max_epochs=max_epochs,
        patience=patience
    )

    train_pred = predict_lstm(model, X_train)
    val_pred = predict_lstm(model, X_val)
    test_pred = predict_lstm(model, X_test)

    train_eval = evaluate_regression(y_train, train_pred, f"{label} - Train")
    val_eval = evaluate_regression(y_val, val_pred, f"{label} - Validation")
    test_eval = evaluate_regression(y_test, test_pred, f"{label} - Test")

    train_eval_df = meta_train.copy()
    train_eval_df["prediction"] = train_pred

    val_eval_df = meta_val.copy()
    val_eval_df["prediction"] = val_pred

    test_eval_df = meta_test.copy()
    test_eval_df["prediction"] = test_pred

    tail_eval = evaluate_top_decile(
        test_eval_df,
        pred_col="prediction",
        target_col=target_col,
        label=f"{label} - Test Top Decile"
    )

    return {
        "model": model,
        "best_val_loss": best_val_loss,
        "train_eval": train_eval,
        "val_eval": val_eval,
        "test_eval": test_eval,
        "tail_eval": tail_eval,
        "train_eval_df": train_eval_df,
        "val_eval_df": val_eval_df,
        "test_eval_df": test_eval_df
    }

# Set common LSTM hyperparameters
seq_len = 7
hidden_size = 64
num_layers = 1
dropout = 0.0
lr = 1e-3
batch_size = 256
max_epochs = 100
patience = 10

# Run baseline LSTM
baseline_results = run_lstm_experiment(
    train_inner_df=train_inner,
    val_df=val_inner,
    test_df=test_lstm,
    feature_cols=baseline_feature_cols,
    target_col=target_col,
    label="LSTM Baseline",
    seq_len=seq_len,
    hidden_size=hidden_size,
    num_layers=num_layers,
    dropout=dropout,
    lr=lr,
    batch_size=batch_size,
    max_epochs=max_epochs,
    patience=patience
)

# Run wildfire-augmented LSTM
augmented_results = run_lstm_experiment(
    train_inner_df=train_inner,
    val_df=val_inner,
    test_df=test_lstm,
    feature_cols=augmented_feature_cols,
    target_col=target_col,
    label="LSTM + Wildfire",
    seq_len=seq_len,
    hidden_size=hidden_size,
    num_layers=num_layers,
    dropout=dropout,
    lr=lr,
    batch_size=batch_size,
    max_epochs=max_epochs,
    patience=patience
)

# Build result tables
overall_comparison = pd.DataFrame([
    baseline_results["train_eval"],
    baseline_results["val_eval"],
    baseline_results["test_eval"],
    augmented_results["train_eval"],
    augmented_results["val_eval"],
    augmented_results["test_eval"]
])

tail_comparison = pd.DataFrame([
    baseline_results["tail_eval"],
    augmented_results["tail_eval"]
])

# Print key outputs
print("\nBaseline best validation loss:")
print(baseline_results["best_val_loss"])

print("\nAugmented best validation loss:")
print(augmented_results["best_val_loss"])

print("\nOverall performance comparison:")
display(overall_comparison)

print("Top-decile performance comparison:")
display(tail_comparison)

Using device: cuda
GPU name: NVIDIA GeForce RTX 3090
Train inner shape: (11866, 69)
Validation shape: (2961, 69)
Test shape: (3696, 69)
Validation cutoff date: 2023-11-12 00:00:00

LSTM Baseline sequence shapes:
X_train: (11740, 7, 34) X_val: (2835, 7, 34) X_test: (3570, 7, 34)
Epoch 001 | Train Loss: 2280.2445 | Val Loss: 1512.0076
Epoch 010 | Train Loss: 542.7405 | Val Loss: 163.8371
Epoch 020 | Train Loss: 371.5081 | Val Loss: 140.0579
Epoch 030 | Train Loss: 315.0581 | Val Loss: 120.8305
Epoch 040 | Train Loss: 274.8065 | Val Loss: 110.8891
Epoch 050 | Train Loss: 253.8998 | Val Loss: 115.1175
Early stopping triggered at epoch 56.

LSTM + Wildfire sequence shapes:
X_train: (11740, 7, 38) X_val: (2835, 7, 38) X_test: (3570, 7, 38)
Epoch 001 | Train Loss: 2255.8824 | Val Loss: 1392.5289
Epoch 010 | Train Loss: 537.5893 | Val Loss: 161.9043
Epoch 020 | Train Loss: 369.9051 | Val Loss: 159.0028
Early stopping triggered at epoch 22.

Baseline best validation loss:
109.37108929951985

Au

,model,RMSE,MAE
0,LSTM Baseline - Train,16.030834,9.578955
1,LSTM Baseline - Validation,10.845120,8.435440
2,LSTM Baseline - Test,12.258577,9.278599
3,LSTM + Wildfire - Train,20.859851,13.122096
4,LSTM + Wildfire - Validation,12.334731,9.966434
5,LSTM + Wildfire - Test,15.104212,11.419574


Top-decile performance comparison:


,model,n_top_decile,top_decile_RMSE,top_decile_MAE
0,LSTM Baseline - Test Top Decile,390,22.668928,19.491741
1,LSTM + Wildfire - Test Top Decile,390,34.553632,32.881744


Note that this first version of LSTM show significant underfitting, so we tune the hyperparameters to get a better performance.

In [32]:
# Define a small hyperparameter grid for additional LSTM refinement
configs = [
    {"seq_len": 7,  "hidden_size": 128, "num_layers": 1, "dropout": 0.0, "lr": 3e-4},
    {"seq_len": 7,  "hidden_size": 128, "num_layers": 2, "dropout": 0.2, "lr": 3e-4},
    {"seq_len": 14, "hidden_size": 128, "num_layers": 1, "dropout": 0.0, "lr": 3e-4},
    {"seq_len": 14, "hidden_size": 128, "num_layers": 2, "dropout": 0.2, "lr": 3e-4},
    # Include the previous stronger configuration for comparison
    {"seq_len": 14, "hidden_size": 128, "num_layers": 2, "dropout": 0.2, "lr": 5e-4},
]

batch_size = 256
max_epochs = 200
patience = 20

search_results = []

for i, cfg in enumerate(configs, start=1):
    print("\n" + "=" * 100)
    print(f"Running config {i}/{len(configs)}: {cfg}")
    print("=" * 100)

    # Run baseline LSTM for the current configuration
    baseline_res = run_lstm_experiment(
        train_inner_df=train_inner,
        val_df=val_inner,
        test_df=test_lstm,
        feature_cols=baseline_feature_cols,
        target_col=target_col,
        label=f"LSTM Baseline | cfg_{i}",
        seq_len=cfg["seq_len"],
        hidden_size=cfg["hidden_size"],
        num_layers=cfg["num_layers"],
        dropout=cfg["dropout"],
        lr=cfg["lr"],
        batch_size=batch_size,
        max_epochs=max_epochs,
        patience=patience
    )

    # Run wildfire-augmented LSTM for the current configuration
    wildfire_res = run_lstm_experiment(
        train_inner_df=train_inner,
        val_df=val_inner,
        test_df=test_lstm,
        feature_cols=augmented_feature_cols,
        target_col=target_col,
        label=f"LSTM + Wildfire | cfg_{i}",
        seq_len=cfg["seq_len"],
        hidden_size=cfg["hidden_size"],
        num_layers=cfg["num_layers"],
        dropout=cfg["dropout"],
        lr=cfg["lr"],
        batch_size=batch_size,
        max_epochs=max_epochs,
        patience=patience
    )

    # Store baseline results
    search_results.append({
        "config_id": i,
        "model_type": "baseline",
        "seq_len": cfg["seq_len"],
        "hidden_size": cfg["hidden_size"],
        "num_layers": cfg["num_layers"],
        "dropout": cfg["dropout"],
        "lr": cfg["lr"],
        "best_val_loss": baseline_res["best_val_loss"],
        "val_RMSE": baseline_res["val_eval"]["RMSE"],
        "val_MAE": baseline_res["val_eval"]["MAE"],
        "test_RMSE": baseline_res["test_eval"]["RMSE"],
        "test_MAE": baseline_res["test_eval"]["MAE"],
        "top_decile_RMSE": baseline_res["tail_eval"]["top_decile_RMSE"],
        "top_decile_MAE": baseline_res["tail_eval"]["top_decile_MAE"],
        "result_obj": baseline_res
    })

    # Store wildfire results
    search_results.append({
        "config_id": i,
        "model_type": "wildfire",
        "seq_len": cfg["seq_len"],
        "hidden_size": cfg["hidden_size"],
        "num_layers": cfg["num_layers"],
        "dropout": cfg["dropout"],
        "lr": cfg["lr"],
        "best_val_loss": wildfire_res["best_val_loss"],
        "val_RMSE": wildfire_res["val_eval"]["RMSE"],
        "val_MAE": wildfire_res["val_eval"]["MAE"],
        "test_RMSE": wildfire_res["test_eval"]["RMSE"],
        "test_MAE": wildfire_res["test_eval"]["MAE"],
        "top_decile_RMSE": wildfire_res["tail_eval"]["top_decile_RMSE"],
        "top_decile_MAE": wildfire_res["tail_eval"]["top_decile_MAE"],
        "result_obj": wildfire_res
    })

# Build a summary table
lstm_search_df = pd.DataFrame(search_results)

summary_cols = [
    "config_id", "model_type", "seq_len", "hidden_size", "num_layers", "dropout", "lr",
    "best_val_loss", "val_RMSE", "val_MAE", "test_RMSE", "test_MAE",
    "top_decile_RMSE", "top_decile_MAE"
]

print("\nFull LSTM search results:")
display(lstm_search_df[summary_cols].sort_values(["model_type", "best_val_loss", "test_RMSE"]))

# Select the best baseline configuration by validation RMSE
best_baseline_row = (
    lstm_search_df[lstm_search_df["model_type"] == "baseline"]
    .sort_values(["val_RMSE", "test_RMSE", "top_decile_RMSE"])
    .iloc[0]
)

# Select the best wildfire configuration by validation RMSE
best_wildfire_row = (
    lstm_search_df[lstm_search_df["model_type"] == "wildfire"]
    .sort_values(["val_RMSE", "test_RMSE", "top_decile_RMSE"])
    .iloc[0]
)

print("\nBest baseline configuration:")
display(pd.DataFrame([best_baseline_row[summary_cols]]))

print("Best wildfire configuration:")
display(pd.DataFrame([best_wildfire_row[summary_cols]]))

# Extract the stored full result objects for later use
best_baseline_results = best_baseline_row["result_obj"]
best_wildfire_results = best_wildfire_row["result_obj"]

# Build comparison tables for the best selected baseline and wildfire models
best_overall_comparison = pd.DataFrame([
    best_baseline_results["train_eval"],
    best_baseline_results["val_eval"],
    best_baseline_results["test_eval"],
    best_wildfire_results["train_eval"],
    best_wildfire_results["val_eval"],
    best_wildfire_results["test_eval"]
])

best_tail_comparison = pd.DataFrame([
    best_baseline_results["tail_eval"],
    best_wildfire_results["tail_eval"]
])

print("\nOverall comparison for the selected best baseline and wildfire LSTM models:")
display(best_overall_comparison)

print("Top-decile comparison for the selected best baseline and wildfire LSTM models:")
display(best_tail_comparison)


Running config 1/5: {'seq_len': 7, 'hidden_size': 128, 'num_layers': 1, 'dropout': 0.0, 'lr': 0.0003}

LSTM Baseline | cfg_1 sequence shapes:
X_train: (11740, 7, 34) X_val: (2835, 7, 34) X_test: (3570, 7, 34)
Epoch 001 | Train Loss: 2299.6565 | Val Loss: 1597.0784
Epoch 010 | Train Loss: 844.1114 | Val Loss: 362.5567
Epoch 020 | Train Loss: 443.8481 | Val Loss: 142.7231
Epoch 030 | Train Loss: 386.3562 | Val Loss: 198.7548
Epoch 040 | Train Loss: 341.5196 | Val Loss: 126.5817
Epoch 050 | Train Loss: 303.4823 | Val Loss: 119.2427
Epoch 060 | Train Loss: 280.5725 | Val Loss: 111.8827
Epoch 070 | Train Loss: 265.6665 | Val Loss: 109.7872
Epoch 080 | Train Loss: 255.0607 | Val Loss: 116.2135
Epoch 090 | Train Loss: 243.9834 | Val Loss: 115.1772
Epoch 100 | Train Loss: 229.4301 | Val Loss: 112.6419
Early stopping triggered at epoch 106.

LSTM + Wildfire | cfg_1 sequence shapes:
X_train: (11740, 7, 38) X_val: (2835, 7, 38) X_test: (3570, 7, 38)
Epoch 001 | Train Loss: 2313.8915 | Val Loss: 

,config_id,model_type,seq_len,hidden_size,num_layers,dropout,lr,best_val_loss,val_RMSE,val_MAE,test_RMSE,test_MAE,top_decile_RMSE,top_decile_MAE
2,2,baseline,7,128,2,0.2,0.0003,103.933070,10.572323,8.403117,11.786356,8.939309,23.453293,20.651726
0,1,baseline,7,128,1,0.0,0.0003,108.021728,10.773825,8.393353,12.701401,9.716170,23.559417,20.496468
4,3,baseline,14,128,1,0.0,0.0003,112.990132,10.785794,8.375231,12.439240,9.352189,25.409121,22.853844
8,5,baseline,14,128,2,0.2,0.0005,113.543790,10.758787,8.506877,11.965106,9.095653,22.914996,19.809961
6,4,baseline,14,128,2,0.2,0.0003,151.337736,12.416663,10.061213,15.219903,11.613601,34.282044,32.593887
1,1,wildfire,7,128,1,0.0,0.0003,102.745200,10.502313,8.277207,11.889401,9.020151,23.829374,21.044420
3,2,wildfire,7,128,2,0.2,0.0003,104.049322,10.582698,8.141591,11.675581,8.716072,23.531045,20.583334
5,3,wildfire,14,128,1,0.0,0.0003,117.986536,11.023455,8.589640,12.968471,9.752109,27.045560,24.776556
7,4,wildfire,14,128,2,0.2,0.0003,151.297823,12.409983,10.094630,15.168653,11.594076,34.114131,32.417229
9,5,wildfire,14,128,2,0.2,0.0005,627.264486,25.406769,22.190258,29.887091,26.127851,55.240583,54.209087



Best baseline configuration:


,config_id,model_type,seq_len,hidden_size,num_layers,dropout,lr,best_val_loss,val_RMSE,val_MAE,test_RMSE,test_MAE,top_decile_RMSE,top_decile_MAE
2,2,baseline,7,128,2,0.2,0.0003,103.93307,10.572323,8.403117,11.786356,8.939309,23.453293,20.651726


Best wildfire configuration:


,config_id,model_type,seq_len,hidden_size,num_layers,dropout,lr,best_val_loss,val_RMSE,val_MAE,test_RMSE,test_MAE,top_decile_RMSE,top_decile_MAE
1,1,wildfire,7,128,1,0.0,0.0003,102.7452,10.502313,8.277207,11.889401,9.020151,23.829374,21.04442



Overall comparison for the selected best baseline and wildfire LSTM models:


,model,RMSE,MAE
0,LSTM Baseline | cfg_2 - Train,16.849347,9.892792
1,LSTM Baseline | cfg_2 - Validation,10.572323,8.403117
2,LSTM Baseline | cfg_2 - Test,11.786356,8.939309
3,LSTM + Wildfire | cfg_1 - Train,15.875338,9.618842
4,LSTM + Wildfire | cfg_1 - Validation,10.502313,8.277207
5,LSTM + Wildfire | cfg_1 - Test,11.889401,9.020151


Top-decile comparison for the selected best baseline and wildfire LSTM models:


,model,n_top_decile,top_decile_RMSE,top_decile_MAE
0,LSTM Baseline | cfg_2 - Test Top Decile,390,23.453293,20.651726
1,LSTM + Wildfire | cfg_1 - Test Top Decile,390,23.829374,21.044420


We can see that under the best tuned hyperparameters, the LSTM model shows pretty good performance. The baseline LSTM model performance has got close to the OLS baseline model. Also, after adding wildfire features, LSTM has got better overall performance, but did not improve the performance in predicting extreme AQI days.

### Overall comparison between models

In [43]:
# Build two separate tables:
# 1. Overall performance table sorted by overall RMSE
# 2. Extreme-day performance table sorted by top-decile RMSE

overall_rows = []
extreme_rows = []

# OLS results
overall_rows.append({
    "model_family": "OLS",
    "spec": "Baseline",
    "RMSE": baseline_test_metrics["RMSE"],
    "MAE": baseline_test_metrics["MAE"]
})
overall_rows.append({
    "model_family": "OLS",
    "spec": "+ Wildfire",
    "RMSE": aug_test_metrics["RMSE"],
    "MAE": aug_test_metrics["MAE"]
})

extreme_rows.append({
    "model_family": "OLS",
    "spec": "Baseline",
    "top_decile_RMSE": baseline_tail_metrics["top_decile_RMSE"],
    "top_decile_MAE": baseline_tail_metrics["top_decile_MAE"]
})
extreme_rows.append({
    "model_family": "OLS",
    "spec": "+ Wildfire",
    "top_decile_RMSE": aug_tail_metrics["top_decile_RMSE"],
    "top_decile_MAE": aug_tail_metrics["top_decile_MAE"]
})

# NN results
overall_rows.append({
    "model_family": "NN",
    "spec": "Baseline",
    "RMSE": nn_baseline_results["test_eval"]["RMSE"],
    "MAE": nn_baseline_results["test_eval"]["MAE"]
})
overall_rows.append({
    "model_family": "NN",
    "spec": "+ Wildfire",
    "RMSE": nn_augmented_results["test_eval"]["RMSE"],
    "MAE": nn_augmented_results["test_eval"]["MAE"]
})

extreme_rows.append({
    "model_family": "NN",
    "spec": "Baseline",
    "top_decile_RMSE": nn_baseline_results["tail_eval"]["top_decile_RMSE"],
    "top_decile_MAE": nn_baseline_results["tail_eval"]["top_decile_MAE"]
})
extreme_rows.append({
    "model_family": "NN",
    "spec": "+ Wildfire",
    "top_decile_RMSE": nn_augmented_results["tail_eval"]["top_decile_RMSE"],
    "top_decile_MAE": nn_augmented_results["tail_eval"]["top_decile_MAE"]
})

# LSTM cfg_2 results
lstm_cfg2_baseline = lstm_search_df[
    (lstm_search_df["config_id"] == 2) & (lstm_search_df["model_type"] == "baseline")
].iloc[0]

lstm_cfg2_wildfire = lstm_search_df[
    (lstm_search_df["config_id"] == 2) & (lstm_search_df["model_type"] == "wildfire")
].iloc[0]

overall_rows.append({
    "model_family": "LSTM",
    "spec": "Baseline (cfg_2)",
    "RMSE": lstm_cfg2_baseline["test_RMSE"],
    "MAE": lstm_cfg2_baseline["test_MAE"]
})
overall_rows.append({
    "model_family": "LSTM",
    "spec": "+ Wildfire (cfg_2)",
    "RMSE": lstm_cfg2_wildfire["test_RMSE"],
    "MAE": lstm_cfg2_wildfire["test_MAE"]
})

extreme_rows.append({
    "model_family": "LSTM",
    "spec": "Baseline (cfg_2)",
    "top_decile_RMSE": lstm_cfg2_baseline["top_decile_RMSE"],
    "top_decile_MAE": lstm_cfg2_baseline["top_decile_MAE"]
})
extreme_rows.append({
    "model_family": "LSTM",
    "spec": "+ Wildfire (cfg_2)",
    "top_decile_RMSE": lstm_cfg2_wildfire["top_decile_RMSE"],
    "top_decile_MAE": lstm_cfg2_wildfire["top_decile_MAE"]
})

overall_table = pd.DataFrame(overall_rows).sort_values(["RMSE", "MAE"]).reset_index(drop=True)
extreme_table = pd.DataFrame(extreme_rows).sort_values(["top_decile_RMSE", "top_decile_MAE"]).reset_index(drop=True)

print("Overall performance comparison:")
display(overall_table)

print("Extreme-day performance comparison:")
display(extreme_table)

Overall performance comparison:


,model_family,spec,RMSE,MAE
0,LSTM,+ Wildfire (cfg_2),11.675581,8.716072
1,OLS,+ Wildfire,11.704901,8.697979
2,OLS,Baseline,11.726270,8.682664
3,LSTM,Baseline (cfg_2),11.786356,8.939309
4,NN,Baseline,12.300621,9.295328
5,NN,+ Wildfire,12.782537,9.542219


Extreme-day performance comparison:


,model_family,spec,top_decile_RMSE,top_decile_MAE
0,NN,Baseline,21.523922,18.380774
1,OLS,Baseline,22.470814,19.058420
2,NN,+ Wildfire,22.558778,19.144850
3,OLS,+ Wildfire,22.638192,19.316394
4,LSTM,Baseline (cfg_2),23.453293,20.651726
5,LSTM,+ Wildfire (cfg_2),23.531045,20.583334


We find that the LSTM model achieves overall test performance close to OLS, while the NN model performs worse in overall accuracy. However, the NN baseline performs best on extreme AQI days. Adding wildfire features slightly improves overall performance for OLS and LSTM, but for all three model families, it leads to slightly worse performance on extreme AQI days.

Although LSTM is competitive with OLS in overall forecasting, the current results do not support a strong claim that it would outperform OLS if more data were added. A more cautious interpretation is that sequence models such as LSTM may benefit from a larger training sample, but this remains a possibility rather than a conclusion from the present analysis.